# System research notebook

Copied verbatim from `research/systems/_template/template.ipynb` by `quantlab.systems.create_system` (DESIGN §1/§3/§10 Phase 2). Every stage cell below is a thin wrapper over `quantlab`; heavy logic lives in the library, not here. All of it reads the `SYSTEM`/`CARD` dicts built in §0 from this folder's own `hypothesis.md` -- nothing below is specific to one system, including this heading; the system's name/slug is shown once §0 runs.

## 0. Setup

In [ ]:
import sys
from pathlib import Path

# Jupyter's default working directory is the notebook's own folder; nbconvert callers should
# pass resources={"metadata": {"path": <this folder>}} to ExecutePreprocessor if that isn't
# already true. hypothesis.md lives right next to this notebook.
SYSTEM_DIR = Path.cwd()

# #33: `quantlab` is not pip-installed, and a fresh kernel subprocess (nbconvert/nbclient) has no
# PYTHONPATH of its own -- walk up from this folder to whichever ancestor directory holds the
# `quantlab` package (the repo root) and put that on sys.path, so the import below resolves
# whatever the kernel's cwd or install, with no environment setup required.
_root = SYSTEM_DIR
while not (_root / "quantlab" / "__init__.py").is_file():
    if _root.parent == _root:
        raise RuntimeError(f"could not find a 'quantlab' package in any parent of {SYSTEM_DIR}")
    _root = _root.parent
if str(_root) not in sys.path:
    sys.path.insert(0, str(_root))

from quantlab import config
from quantlab.systems import read_card

CARD = read_card(SYSTEM_DIR / "hypothesis.md")
SYSTEM = {
    "book": CARD["book"],
    "issue": CARD["issue"],
    "slug": CARD["slug"],
    "name": CARD.get("name", CARD["slug"]),
    "risk_type": CARD.get("risk_type"),
    "status": CARD.get("status", "testing"),
    # Set by strategy-engineer once the instrument/timeframe is picked (S3): add `symbol:` /
    # `timeframe:` (and optionally `start:` / `end:`) to hypothesis.md's front matter.
    "symbol": CARD.get("symbol"),
    "timeframe": CARD.get("timeframe"),
    "start": CARD.get("start"),
    "end": CARD.get("end"),
    # Set by S4 (quantlab.systems.set_front_matter) the first time it resolves this system's
    # study -- read here so a later stage / a later "Run all" doesn't have to re-scan the ledger.
    "study_id": CARD.get("study_id"),
}
LEDGER_DIR = config.LEDGER_DIR
STUDIES_DIR = config.STUDIES_DIR
RESULTS_DIR = SYSTEM_DIR / "results"
RESULTS_DIR.mkdir(parents=True, exist_ok=True)
SYSTEM

## 1. Hypothesis

In [ ]:
from IPython.display import Markdown, display

display(Markdown(f"**Status:** {SYSTEM['status']}  ·  **Risk type:** {SYSTEM['risk_type']}  ·  "
                 f"**Issue:** #{SYSTEM['issue']}"))
display(Markdown(CARD.get("body", "")))

## 2. Implementation

In [ ]:
import inspect
from dataclasses import asdict

from quantlab.costs import InstrumentSpec
from quantlab.strategies import get_strategy
from quantlab.systems import card_space_corners
from quantlab.testing import (
    assert_engine_causal,
    assert_no_lookahead,
    assert_strategy_source_clean,
    synthetic_bars,
)

StrategyCls = get_strategy(SYSTEM["slug"])
assert_strategy_source_clean(inspect.getfile(StrategyCls))

# Same class/Params resolution order as quantlab.evaluators.RuleEvaluator.build_strategy: a
# strategy is built from its declared prior (default) parameters here, so this audit exercises
# the real class with the card's own prior values, not an arbitrary stand-in.
_params_cls = getattr(StrategyCls, "params_cls", None) or getattr(StrategyCls, "Params", None)

# #31: audit the prior AND every corner of the card's pre-registered grid, when it declares one
# parseable this way (opt.SearchSpace itself isn't built until S4) -- {} = the prior (no override).
_param_overrides = [{}]
if _params_cls is not None:
    _param_overrides += card_space_corners(CARD)

_audit_bars = synthetic_bars(600, seed=0, timeframe="H1")
_audit_spec = InstrumentSpec(
    symbol="AUDIT", digits=5, point=1e-5, contract_size=100_000.0, tick_size=1e-5,
    volume_min=0.01, volume_step=0.01, volume_max=100.0, base_ccy="EUR", quote_ccy="USD",
    swap_mode="points", swap_long=0.0, swap_short=0.0, swap_3day=2,
    commission_per_lot_rt=0.0, stops_level=0.0, calibrated=True,
)
for _override in _param_overrides:
    if _params_cls is not None:
        _factory = lambda _o=_override: StrategyCls(_params_cls(**{**asdict(_params_cls()), **_o}))
    else:
        _factory = StrategyCls
    assert_no_lookahead(_factory, _audit_bars)
    assert_engine_causal(_factory(), _audit_bars, _audit_spec, n_checks=8, seed=1, min_history=60)

print(f"{StrategyCls.name}: look-ahead audit clean at the prior and {len(_param_overrides) - 1} "
     f"grid corner(s) (risk_type={StrategyCls.risk_type})")

## 3. Baseline

In [ ]:
if SYSTEM.get("symbol") and SYSTEM.get("timeframe"):
    from quantlab.evaluators import RuleEvaluator, edge_breakdown
    from quantlab.systems import log_baseline

    evaluator = RuleEvaluator(
        StrategyCls, symbol=SYSTEM["symbol"], timeframe=SYSTEM["timeframe"], book=SYSTEM["book"],
        start=SYSTEM.get("start"), end=SYSTEM.get("end"),
    )
    baseline = evaluator({})   # {} = the card's prior (default) parameters, costs on
    print(evaluator.describe())
    print(baseline.metrics)                              # #24: explicit -- this is the `if`
                                                            # branch, so auto-display never fires
    edge = edge_breakdown(baseline, cost=evaluator.cost)   # #25: gross/net/spread/swap/slippage
    print(edge)                                            # per trade, trades/year, cost % gross

    # #28: an audit-trail event, not a new study row -- see quantlab.systems.log_baseline.
    _params_used = _params_cls().as_dict() if _params_cls is not None else {}
    _logged = log_baseline(
        SYSTEM["book"], SYSTEM["issue"], SYSTEM["slug"],
        params=_params_used, edge_breakdown=edge, cost_model_version=evaluator.cost_version,
    )
    if _logged is None:
        print("S3: no study in the ledger yet for this system -- baseline not logged to the "
             "ledger (re-run this cell after S4; #35, S3 is independently re-runnable), or a "
             "'baseline' event is already logged for it.")
    else:
        print(f"S3: baseline logged to the ledger under {_logged['study_id']}")
else:
    evaluator = None
    baseline = None
    edge = None
    print("S3 pending: add `symbol:` / `timeframe:` to hypothesis.md's front matter, then re-run.")

## 4. Optimisation

In [ ]:
if evaluator is not None:
    from quantlab import opt
    from quantlab.systems import find_study_id, set_front_matter

    # S4 (optimization-architect): declare the real search space here -- every numeric Param
    # needs plateau_scale="relative" or plateau_step=... (DESIGN §4.2, pre-registered on the
    # card) -- then run_study(...). Left unset in the template so it never launches a study with
    # placeholder bounds.
    space = None   # opt.SearchSpace((opt.IntParam(...), ...))

    # #35: a full "Run all" must not repeat a logged run_study -- load the ledger's own study
    # (found by id, else by a book/issue/slug ledger lookup) instead of re-running it blind.
    study_id = SYSTEM.get("study_id") or find_study_id(SYSTEM["book"], SYSTEM["issue"], SYSTEM["slug"])
    if study_id is not None:
        study = opt.load_study(study_id)
        print(f"S4: loaded existing study {study_id} read-only (run_study skipped, #35)")
    elif space is not None:
        study = opt.run_study(
            evaluator, space, book=SYSTEM["book"], system=SYSTEM["slug"], issue=SYSTEM["issue"], attempt=1,
        )
        study_id = study.study_id
        print(f"S4: ran a new study {study_id}")
    else:
        study = None
        print("S4 pending: no search space declared yet.")

    if study_id is not None and CARD.get("study_id") != study_id:
        set_front_matter(SYSTEM_DIR / "hypothesis.md", study_id=study_id)   # #35
        CARD["study_id"] = study_id
    SYSTEM["study_id"] = study_id
else:
    space = None
    study = None
    study_id = None
    print("S4 pending: needs a baseline (S3) first.")

## 5. Validation

In [ ]:
if evaluator is not None:
    import polars as pl

    from quantlab import gates, ledger, opt
    from quantlab.systems import find_study_id

    # Resolved independently of §4's own `study` variable (#35/#68: this cell must work even if
    # §4 did not run this kernel session -- only the ledger/trial-store, loaded by id, are trusted).
    study_id = SYSTEM.get("study_id") or find_study_id(SYSTEM["book"], SYSTEM["issue"], SYSTEM["slug"])
    study = opt.load_study(study_id) if study_id is not None else None

    if study is None:
        gate_report = None
        gate_table = None
        print("S5 pending: needs an optimisation study (S4) first.")
    elif ledger.study_events(study.study_id, "gates"):
        gate_report = None
        gate_table = None
        print(f"S5: {study.study_id} already has a logged gates event -- "
             "evaluate_gates(log=True) skipped (#35: a full 'Run all' must never log it twice).")
    else:
        gate_report = gates.evaluate_gates(study, evaluator, periods_per_year=evaluator.periods_per_year, log=True)
        gate_table = pl.DataFrame([
            {"gate": r.gate, "value": r.display if r.display is not None else r.value,
             "threshold": r.threshold, "status": r.status, "interpretation": r.interpretation}
            for r in gate_report.rows
        ])
    gate_table
else:
    study = None
    gate_report = None
    gate_table = None
    print("S5 pending: needs a baseline (S3) first.")

## 6. Red-team findings

In [ ]:
FINDINGS_PATH = RESULTS_DIR / "redteam_findings.md"
if FINDINGS_PATH.exists():
    from IPython.display import Markdown, display

    display(Markdown(FINDINGS_PATH.read_text()))
else:
    print(f"S6 pending: no findings file yet at {FINDINGS_PATH}")

## 7. Report

In [ ]:
if evaluator is not None:
    from quantlab import ledger, report
    from quantlab.systems import find_study_id

    # #68: resolved independently of §4/§5's own variables -- report.tear_sheet only needs a
    # study_id + evaluator, reading everything else (including the gates event) from the ledger
    # and trial store itself, so this cell works even run alone in a fresh kernel.
    study_id = SYSTEM.get("study_id") or find_study_id(SYSTEM["book"], SYSTEM["issue"], SYSTEM["slug"])
    if study_id is not None and ledger.study_events(study_id, "gates"):
        ts = report.tear_sheet(study_id, evaluator=evaluator, risk_type=SYSTEM.get("risk_type"))
        ts.write_results(RESULTS_DIR)
        report.write_card(
            ts, slug=SYSTEM["slug"], name=SYSTEM["name"], idea=CARD.get("idea", ""),
            status=SYSTEM.get("status", "testing"), issue=SYSTEM["issue"], book=SYSTEM["book"],
        )
    else:
        ts = None
        print("S7 pending: needs validation (S5, a logged gates event) first.")
else:
    ts = None
    print("S7 pending: needs a baseline (S3) first.")
ts.headline if ts is not None else None

## 8. Holdout

In [ ]:
# Status only. The holdout is unlocked once, by the user, via `/unlock-holdout <system>` -- never
# from this notebook. This cell never calls quantlab.data with include_holdout=True.
print(f"S8 holdout: system status = {SYSTEM.get('status')!r} (book {SYSTEM['book']}). "
     "Run `/unlock-holdout` yourself when checkpoint B is reached (DESIGN §3, §4.4).")